<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DROID Multi-View 3D Tracking Pipeline

This notebook is a **thin orchestration layer** — all algorithm code lives in the GitHub repo.

**Four stages** controlled by global flags (Stages 1–3 can compute or load from GCS):

| Stage | Compute | Load from GCS | Output |
|---|---|---|---|
| 1. Depth | `compute_depth.py` | `gs://dm-tapnet/tmp/droid/depth/` | Stereo depth + gripper refinement |
| 2. Extrinsics | `compute_extrinsics.py` | `gs://dm-tapnet/tmp/droid/extrinsics/` | Camera-robot alignment |
| 3. Tracks | `compute_tracks.py` | `gs://dm-tapnet/tmp/droid/tracks/` | Static BG + Robot 3D tracks |
| 4. Metrics | `compute_metrics.py` | — | Quality metrics + visualization |

---
## 0. Environment Setup

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
  subprocess.run(
    ["git", "clone", "--recursive", "https://github.com/yangyi02/droid.git", "/content/droid"],
    check=True,
  )
  os.chdir("/content/droid")

REPO_DIR = os.getcwd()
CACHE_DIR = os.path.join(REPO_DIR, "data", "cache")
GCS_OUTPUT = "gs://dm-tapnet/tmp/droid"
print(f"cache: {CACHE_DIR}")

In [ ]:
COMPUTE_DEPTH = False
COMPUTE_EXTRINSICS = True
COMPUTE_TRACKS = True

print(f"COMPUTE_DEPTH      = {COMPUTE_DEPTH}")
print(f"COMPUTE_EXTRINSICS = {COMPUTE_EXTRINSICS}")
print(f"COMPUTE_TRACKS     = {COMPUTE_TRACKS}")

In [ ]:
if IN_COLAB:
  from google.colab import auth
  auth.authenticate_user()

  subprocess.run([sys.executable, "-m", "pip", "install", "-U", "ipython"], check=True)

  subprocess.run(["bash", "setup.sh"] + ([] if COMPUTE_DEPTH else ["--no-depth"]), check=True)
else:
  print("[SKIP] local checkout — dependencies come from setup.sh / the venv")

In [ ]:
import random

import cv2
import matplotlib.pyplot as plt
import mediapy as media
import numpy as np
import torch

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
%reload_ext autoreload
%autoreload 2

from config import get_config
import core.depth
import core.geometry
import core.io
import core.physics
import core.tracking
import core.visualization
import compute_depth
import compute_extrinsics
import compute_metrics
import compute_tracks

config = get_config()

In [ ]:
serials_db, id_to_path, keep_ranges, extrinsics_db, _ = core.io.load_metadata(config)

with open(os.path.join(REPO_DIR, "episodes_success.txt")) as f:
  valid_ids = sorted(line.strip() for line in f if line.strip())
print(f"{len(valid_ids)} successful episodes to pick from")

In [ ]:
episode_id = random.choice(valid_ids)

print(f"Episode: {episode_id}")

In [ ]:
scene_constants = compute_depth.init_episode(
  episode_id, config.paths.raw, id_to_path, serials_db, keep_ranges
)

print(f"scene_constants initialized: {list(scene_constants['camera'].keys())}")

---
## 1. Stage 1: Depth

Stereo depth via S2M2 + SAM gripper refinement.

In [ ]:
if COMPUTE_DEPTH:
  s2m2_model, sam_predictor, run_stereo_matching = compute_depth.init_all_models()

  scene_constants = compute_depth.extract_svo_video(scene_constants)
  scene_constants = compute_depth.parse_robot_kinematics(scene_constants)
  scene_constants = compute_depth.align_temporal_streams(scene_constants)
  scene_constants = core.depth.compute_stereo_depth(
    scene_constants, s2m2_model, run_stereo_matching, device
  )

  wrist_data = scene_constants["camera"][scene_constants["meta"]["wrist_serial"]]
  wrist_data["original_raw_depth"] = wrist_data["raw_depth"].copy()
  scene_constants = core.depth.build_universal_gripper_mask(scene_constants, sam_predictor)
  scene_constants = core.depth.distill_empirical_gripper_depth(scene_constants)
  scene_constants = core.depth.inject_gripper_depth(scene_constants)

  print("Stage 1 (Depth) COMPUTE complete")
else:
  depth_root = os.path.join(CACHE_DIR, "depth")
  src = f"{GCS_OUTPUT}/depth/{episode_id}"
  dst = os.path.join(depth_root, episode_id)
  subprocess.run(["gcloud", "storage", "rsync", "-r", src, dst], check=True)

  scene_constants = core.io.load_depth_data(
    episode_id, depth_root, load_video="full", inspection=True
  )

  print("Stage 1 LOADED from GCS")

In [ ]:
core.visualization.inspect_dict_structure(scene_constants)

frames = core.visualization.render_multicam_disparity_video(scene_constants, max_frames=30)
media.show_video(frames, fps=10, title="Depth [left | right | disparity] per camera")

In [ ]:
core.visualization.render_gripper_refinement_inspection(scene_constants, frame_idx=0)

cam_data = scene_constants['camera'][scene_constants['meta']['wrist_serial']]
core.visualization.render_distilled_gripper_3d(
  median_depth=cam_data['empirical_gripper_depth'],
  K_mat=cam_data['K_mat'],
  rgb_img=cam_data['video_rgb'][0],
)

---
## 2. Stage 2: Extrinsics

Dataset extrinsics → differentiable robot alignment → global joint optimization.

In [ ]:
pb_renderer = core.physics.PyBulletRenderer(config.paths.urdf, gpu=config.render.gpu)
print(f"PyBulletRenderer EGL: {pb_renderer.gpu}")

if COMPUTE_EXTRINSICS:
  scene_state = compute_extrinsics.init_camera_states(scene_constants, extrinsics_db)

  scene_state = compute_extrinsics.per_camera_alignment(
    scene_constants, pb_renderer, scene_state, device
  )

  scene_state = compute_extrinsics.global_joint_alignment(
    scene_constants, scene_state, pb_renderer, device
  )

  print("Stage 2 (Extrinsics) COMPUTE complete")
else:
  ext_root = os.path.join(CACHE_DIR, "extrinsics")
  src = f"{GCS_OUTPUT}/extrinsics/{episode_id}"
  dst = os.path.join(ext_root, episode_id)
  subprocess.run(["gcloud", "storage", "rsync", "-r", src, dst], check=True)

  scene_state = core.io.load_extrinsics(scene_constants, ext_root)

  print("Extrinsics LOADED from GCS")

In [ ]:
axes_frames = core.visualization.render_cross_camera_axes(
  scene_constants, scene_state, max_frames=30
)
media.show_video(axes_frames, fps=10, title="Camera Axes Overlay")

In [ ]:
seg_frames = core.visualization.render_segmentation_video(
  scene_constants, scene_state, pb_renderer, max_frames=30
)
media.show_video(seg_frames, fps=10, title="Robot Mask Overlay")

In [ ]:
core.visualization.render_fused_point_cloud(
  scene_constants, scene_state, frame_idx=0, height=600, width=1000
)

In [ ]:
orbit_frames = core.visualization.render_4d_orbit_with_tracks(
  scene_constants, scene_state, max_frames=30
)
media.show_video(orbit_frames, fps=10, title="4D Orbit")

---
## 3. Stage 3: Tracking

**Static Background + Robot Tracks** — no tracker model dependency.

**Dual-Track Architecture:**
- **Track A (Static Background)**: Multi-view depth consensus → fixed world 3D → project to 2D per-view using extrinsics (static prior)
- **Track B (Robot)**: URDF forward kinematics → per-link binding → cross-view projection

In [ ]:
if COMPUTE_TRACKS:
  camera_ids = list(scene_constants['camera'].keys())
  T_frames = len(scene_constants['camera'][camera_ids[0]]['video_rgb'])

  robot_masks = compute_tracks.render_robot_masks(scene_constants, scene_state, pb_renderer)

  static_pts_3d, static_rgb = compute_tracks.find_static_candidates(
    scene_constants, scene_state, robot_masks, num_points=config.tracks.num_static_points
  )

  if len(static_pts_3d) > 0:
    static_per_cam_tracks, static_per_cam_vis = compute_tracks.project_static_tracks(
      static_pts_3d, scene_constants, scene_state, robot_masks
    )
  else:
    static_per_cam_tracks = {
      cam: np.zeros((T_frames, 0, 2), dtype=np.float32) for cam in camera_ids
    }
    static_per_cam_vis = {cam: np.zeros((T_frames, 0), dtype=bool) for cam in camera_ids}

  n_static = len(static_pts_3d)

  robot_traj_3d, robot_per_cam_tracks, robot_per_cam_vis, n_robot = (
    compute_tracks.compute_robot_tracks(
      scene_constants,
      scene_state,
      pb_renderer,
      max_robot_pts_per_cam=config.tracks.max_robot_pts_per_cam,
    )
  )

  (final_traj_3d, final_vis_global, final_per_cam_tracks, final_per_cam_vis, n_static, n_robot) = (
    compute_tracks.merge_tracks(
      static_pts_3d,
      static_per_cam_tracks,
      static_per_cam_vis,
      robot_traj_3d,
      robot_per_cam_tracks,
      robot_per_cam_vis,
      camera_ids,
      T_frames,
    )
  )

  print(
    f"\nStage 3 COMPUTE complete: {n_static} static + {n_robot} robot "
    f"= {final_traj_3d.shape[1]} points"
  )

else:
  tracks_root = os.path.join(CACHE_DIR, "tracks")
  src = f"{GCS_OUTPUT}/tracks/{episode_id}"
  dst = os.path.join(tracks_root, episode_id)
  subprocess.run(["gcloud", "storage", "rsync", "-r", src, dst], check=True)

  tracks = compute_metrics.load_track_data(episode_id, tracks_root)

  final_traj_3d = tracks["traj_3d"]
  final_vis_global = tracks["vis_global"]
  final_per_cam_tracks = tracks["per_cam_tracks"]
  final_per_cam_vis = tracks["per_cam_vis"]
  n_static, n_robot = tracks["n_static"], tracks["n_robot"]

  T, N, _ = final_traj_3d.shape
  print(
    f"Stage 3 LOADED from GCS — {len(final_per_cam_tracks)} cameras, "
    f"{T} frames, {n_static} static + {n_robot} robot = {N} points"
  )

In [ ]:
camera_ids = list(scene_constants['camera'].keys())

ref_cam = camera_ids[min(1, len(camera_ids) - 1)]
y_static = final_per_cam_tracks[ref_cam][0, :n_static, 1]
norm_s = plt.Normalize(y_static.min(), y_static.max())
static_colors = (plt.cm.gist_rainbow(norm_s(y_static))[:, :3] * 255).astype(np.uint8)

robot_colors = np.full((n_robot, 3), [255, 50, 50], dtype=np.uint8)
combined_colors = np.concatenate([static_colors, robot_colors], axis=0)

all_frames_static = []
for cam_id in camera_ids:
  cam_data = scene_constants['camera'][cam_id]
  tracks = final_per_cam_tracks[cam_id][:, :n_static, :]
  vis = final_per_cam_vis[cam_id][:, :n_static]
  frames = core.visualization.render_2d_tracking_video(
    cam_data['video_rgb'],
    tracks,
    vis,
    global_colors=static_colors,
    tgt_size=(256, 456),
    linewidth=1,
    max_frames=60,
  )
  for f in frames:
    cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
    cv2.putText(
      f, f"Cam [{cam_id[:8]}]", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1
    )
  all_frames_static.append(np.array(frames))

if all_frames_static:
  combined = np.concatenate(all_frames_static, axis=2)
  media.show_video(
    combined, fps=10, title=f"Static Background Tracks ({n_static} points) — All Cameras"
  )

if n_robot > 0:
  all_frames_robot = []
  for cam_id in camera_ids:
    cam_data = scene_constants['camera'][cam_id]
    tracks = final_per_cam_tracks[cam_id][:, n_static:, :]
    vis = final_per_cam_vis[cam_id][:, n_static:]
    frames = core.visualization.render_2d_tracking_video(
      cam_data['video_rgb'],
      tracks,
      vis,
      global_colors=robot_colors,
      tgt_size=(256, 456),
      linewidth=1,
      max_frames=60,
    )
    for f in frames:
      cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
      cv2.putText(
        f, f"Cam [{cam_id[:8]}]", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1
      )
    all_frames_robot.append(np.array(frames))

  if all_frames_robot:
    combined_robot = np.concatenate(all_frames_robot, axis=2)
    media.show_video(combined_robot, fps=10, title=f"Robot Tracks ({n_robot} points) — All Cameras")

all_frames_both = []
for cam_id in camera_ids:
  cam_data = scene_constants['camera'][cam_id]
  frames = core.visualization.render_2d_tracking_video(
    cam_data['video_rgb'],
    final_per_cam_tracks[cam_id],
    final_per_cam_vis[cam_id],
    global_colors=combined_colors,
    tgt_size=(256, 456),
    linewidth=1,
    max_frames=60,
  )
  for f in frames:
    cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
    cv2.putText(
      f, f"Cam [{cam_id[:8]}]", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1
    )
  all_frames_both.append(np.array(frames))

if all_frames_both:
  combined_both = np.concatenate(all_frames_both, axis=2)
  media.show_video(
    combined_both,
    fps=10,
    title=f"All Tracks ({n_static} static (static) + {n_robot} robot (robot))",
  )

In [ ]:
print(f"Using 'final_traj_3d', shape={final_traj_3d.shape}")

orbit_frames = core.visualization.render_4d_orbit_with_tracks(
  scene_constants,
  scene_state,
  tracks_3d=final_traj_3d,
  max_frames=60,
)
media.show_video(orbit_frames, fps=10, title="4D Orbit — Point Cloud + Tracks + Cameras")

---
## 4. Stage 4: Quality Metrics

In [ ]:
all_metrics = compute_metrics.evaluate_episode(
  scene_constants,
  scene_state,
  device,
  final_traj_3d=final_traj_3d,
  final_per_cam_vis=final_per_cam_vis,
  n_static=n_static,
  n_robot=n_robot,
  compute_extrinsics_metrics=True,
  pb_renderer=pb_renderer,
)

print(f"Episode: {all_metrics['episode_id']}")
print(f"{'=' * 60}")

sections = {
  "Scene": ["site", "robot_id", "n_cameras", "image_resolution", "n_frames"],
  "Motion": ["ee_travel_m", "joint_range_mean_rad", "joint_range_max_rad", "gripper_range"],
  "Extrinsics": [
    "chamfer_12",
    "chamfer_1w",
    "chamfer_2w",
    "chamfer_mean",
    "overlap_12",
    "overlap_1w",
    "overlap_2w",
    "overlap_mean",
    "robot_loss_cam1",
    "robot_loss_cam2",
    "robot_loss_wrist",
  ],
  "Track Depth Consistency": [k for k in sorted(all_metrics) if k.startswith("depth_residual")],
  "Track Visibility": [k for k in sorted(all_metrics) if k.startswith("vis_")],
}

for section_name, keys in sections.items():
  print(f"\n  {section_name}:")
  for k in keys:
    v = all_metrics.get(k, "—")
    if isinstance(v, float):
      print(f"    {k:45s} = {v:.4f}")
    else:
      print(f"    {k:45s} = {v}")

In [ ]:
camera_ids = list(scene_constants['camera'].keys())

errors = compute_metrics.compute_depth_residual_per_camera(
  scene_constants, scene_state, final_traj_3d, final_per_cam_vis, n_static, n_robot
)

plt.rcParams.update(
  {'font.sans-serif': 'DejaVu Sans', 'axes.edgecolor': '#cccccc', 'axes.linewidth': 0.8}
)
fig, axes = plt.subplots(1, len(camera_ids), figsize=(4.8 * len(camera_ids), 3.8), sharey=True)
if len(camera_ids) == 1:
  axes = [axes]

palette = {'static': '#2da44e', 'robot': '#cf222e', 'all': '#0969da'}

for ax, cam_id in zip(axes, camera_ids):
  s_err = errors[cam_id]['static']
  r_err = errors[cam_id]['robot']
  a_err = errors[cam_id]['all']

  if len(s_err):
    med_s = np.median(s_err)
    ax.hist(
      s_err,
      bins=40,
      range=(0, 40),
      alpha=0.4,
      color=palette['static'],
      label=f'Static (Med: {med_s:.1f} mm)',
    )
    ax.axvline(med_s, color=palette['static'], linestyle='--', linewidth=1.2)
  if len(r_err):
    med_r = np.median(r_err)
    ax.hist(
      r_err,
      bins=40,
      range=(0, 40),
      alpha=0.4,
      color=palette['robot'],
      label=f'Robot (Med: {med_r:.1f} mm)',
    )
    ax.axvline(med_r, color=palette['robot'], linestyle='--', linewidth=1.2)
  if len(a_err):
    med_a = np.median(a_err)
    ax.axvline(
      med_a,
      color=palette['all'],
      linestyle='-',
      linewidth=1.6,
      label=f'Overall (Med: {med_a:.1f} mm)',
    )

  ax.set_title(f'Camera [{cam_id[:8]}]', fontsize=11, pad=10, fontweight='bold')
  ax.set_xlabel('Depth Residual Error (mm)', fontsize=9)
  ax.grid(True, linestyle=':', alpha=0.5)
  ax.spines['top'].set_visible(False)
  ax.spines['right'].set_visible(False)
  ax.legend(frameon=True, facecolor='white', framealpha=0.95, fontsize=8)

axes[0].set_ylabel('Observation Count', fontsize=9)
plt.suptitle(
  f'Depth Consistency (Static={n_static}, Robot={n_robot}, Total={final_traj_3d.shape[1]})',
  fontsize=12,
  y=1.03,
)
plt.tight_layout()
plt.show()